# 09 — Process refinery output

Aggregate JODI refinery output for gasoline and diesel and merge the annual system-regime classification.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.jodi import read_secondary_zip, canonicalise_secondary, filter_portugal_fuels, annualise
from portugal_refining_resilience.validation import assert_nonnegative, assert_unique


In [ ]:
raw_zip = PATHS.raw / "jodi" / "world_secondary_csv.zip"
if not raw_zip.exists():
    raise FileNotFoundError("Run notebook 03 first.")
raw = read_secondary_zip(raw_zip)
canonical = canonicalise_secondary(raw)
selected = filter_portugal_fuels(canonical, flows=("refinery output",))
annual = annualise(selected).rename(columns={"product_canonical": "product", "value_kt": "refinery_output_kt"})
annual = annual.loc[annual["year"].between(2005, 2024), ["year", "product", "refinery_output_kt", "source"]]
regime = pd.read_csv(PATHS.processed / "refining_regime_annual.csv")
annual = annual.merge(regime, on="year", how="left", validate="many_to_one")
assert_nonnegative(annual, ["refinery_output_kt"])
assert_unique(annual, ["year", "product"])
persist_dataframe(annual, PATHS.processed / "fuel_refinery_output_annual.csv", key_columns=["year", "product"], metadata={"unit": "kt"})
display(annual.tail())
